# EYES-DEFY-ANEMIA — K-fold re-training: Base tier (4 of 12 combos)

One of **3 tier-specific notebooks** (Base / Mid / Strong), covering the 12 combos that completed the original 18-combo Optuna sweep (the 6 Transformer-family combos OOM'd and are out of scope here). Each notebook re-trains its tier's combos using their own already-found best hyperparameters (no re-tuning) under **3-fold cross-validation** (`Segmentation/scripts/train_pretrained_kfold/kfold_engine.py`), so the reported test-set metric is a mean ± std across 3 independent trainings instead of one point estimate.

**This notebook covers:**
- EfficientNet-B1 U-Net (CNN, ~8.8M), `palpebral`
- EfficientNet-B1 U-Net (CNN, ~8.8M), `forniceal_palpebral`
- CoAtNet-0 U-Net (Hybrid, ~30.8M), `palpebral`
- CoAtNet-0 U-Net (Hybrid, ~30.8M), `forniceal_palpebral`

**Estimated time: ~2.3h worst-case (EfficientNet-B1 ~1.3h + CoAtNet-0 ~1.0h)** — comfortably within one 12h Kaggle session.

**Confirmed settings:** `BATCH_SIZE=32`, `MAX_EPOCHS=250`, `EARLY_STOPPING_PATIENCE=7`, `ReduceLROnPlateau` LR scheduler (factor=0.5, patience=3) — none of trainer_engine.py's original Optuna-sweep settings (30 epochs, patience 5, batch 16, no scheduler) apply here. Full rationale and design decisions: `Segmentation/.project_memory/05_kfold_reevaluation.md`.

Run the 3 tier notebooks independently, in separate Kaggle sessions.

## Setup

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first.
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

Cloning into 'eyes-defy-anemia'...
remote: Enumerating objects: 1699, done.
remote: Counting objects: 100% (1380/1380), done.
remote: Compressing objects: 100% (939/939), done.
remote: Total 1699 (delta 568), reused 1206 (delta 427), pack-reused 319 (from 1)
Receiving objects: 100% (1699/1699), 100.78 MiB | 34.38 MiB/s, done.
Resolving deltas: 100% (731/731), done.
/kaggle/working/eyes-defy-anemia


In [3]:
# Diagnostic, not a hardcoded assumption -- Kaggle's actual dataset mount
# path does not always match its display name, and can be nested deeper
# than expected (this project has been bitten by this before, twice now:
# see classification/.project_memory/kaggle/01_kaggle_notes.md -- real
# paths have landed under /kaggle/input/datasets/<username>/<slug>/, not
# directly under /kaggle/input/<slug>/). Recurses a few levels deep so
# this is caught in one pass instead of needing to descend manually.
# Read the output below, THEN set the dataset dir variable(s) in the next cell.
import os


def print_tree(path, depth=0, max_depth=4):
    for entry in sorted(os.listdir(path)):
        full = os.path.join(path, entry)
        print("  " * depth + entry)
        if os.path.isdir(full) and depth < max_depth:
            print_tree(full, depth + 1, max_depth)


print_tree("/kaggle/input")

datasets
  manivafapour21
    aligned-raw
      aligned_raw
        alignment_log.csv
        images
        masks
    aligned-raw-forniceal
      aligned_raw_forniceal
        alignment_log.csv
        images
        masks


**Before running the next cell:** attach your uploaded dataset(s) to this notebook -- either both zips together as ONE Kaggle dataset, or as TWO SEPARATE datasets (this is what actually happened the first time this notebook was run: Kaggle listed them individually in the Input panel as `aligned_raw` and `aligned_raw_forniceal`). Either way, run the listing cell above, read its REAL printed output, and set the path(s) below from that -- **not** the placeholder text left in by default, and not a guessed path based on the dataset's display name (Kaggle's actual mount path does not always match it).

In [4]:
# Confirmed real mount paths (project author's Kaggle account, verified via
# the print_tree() listing above on 2026-08-08 -- see
# Segmentation/.project_memory/kaggle/01_kaggle_notes.md for the full
# doubly-nested structure this came from). If you re-attach the datasets
# under a different username/slug, or Kaggle changes its mount scheme again,
# re-run the listing cell above and update these two lines from its real
# output -- don't guess.
ALIGNED_RAW_DATASET_DIR = "/kaggle/input/datasets/manivafapour21/aligned-raw"
ALIGNED_RAW_FORNICEAL_DATASET_DIR = "/kaggle/input/datasets/manivafapour21/aligned-raw-forniceal"

In [5]:
# Only packages actually missing from Kaggle's base image (torch/torchvision,
# opencv, pandas, PIL, scikit-learn -- and therefore scipy AND sklearn's
# StratifiedKFold, its own dependency -- are already there; scipy is listed
# explicitly anyway rather than silently assumed, since it is a real,
# load-bearing dependency for HD95 in segmentation_metrics.py. No optuna
# here -- these notebooks re-train with FIXED hyperparameters (each combo's
# own best_params from the earlier Optuna sweep), not a fresh search.
# Deliberately NOT `pip install -r requirements.txt` -- that file is pinned
# to the local Windows/CUDA 13.0 build and would try to reinstall Kaggle's
# own correctly configured GPU PyTorch with an incompatible build.
!pip install -q albumentations scipy segmentation-models-pytorch timm transformers einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.5 MB/s eta 0:00:00


## Data

In [6]:
import shutil
import zipfile
from pathlib import Path

DST_ROOT = Path("Segmentation/data/processed")


def stage_tissue_data(name: str, dataset_dir: str):
    """Copies {name}/ from the given Kaggle-attached dataset directory into
    Segmentation/data/processed/{name}/. Tries three possible layouts rather
    than assuming one, since Kaggle can present an uploaded zip differently
    depending on upload method:
      1. dataset_dir/{name}/images,masks/  -- the zip's own internal "{name}/"
         prefix preserved as-is (this is how aligned_raw.zip/
         aligned_raw_forniceal.zip were actually built -- see
         Segmentation/scripts/build_aligned_dataset{,_forniceal}.py).
      2. dataset_dir/images,masks/         -- Kaggle stripped/flattened that
         top-level folder on extraction.
      3. dataset_dir/{name}.zip            -- never auto-extracted at all,
         still sitting there as a raw zip file.
    """
    dataset_dir = Path(dataset_dir)
    dst = DST_ROOT / name
    shutil.rmtree(dst, ignore_errors=True)

    nested_dir = dataset_dir / name
    flat_zip = dataset_dir / f"{name}.zip"

    if nested_dir.is_dir():
        shutil.copytree(nested_dir, dst)
    elif (dataset_dir / "images").is_dir() and (dataset_dir / "masks").is_dir():
        shutil.copytree(dataset_dir, dst)
    elif flat_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(flat_zip) as zf:
            zf.extractall(DST_ROOT)  # zip's own internal paths already start with f"{name}/"
    else:
        raise FileNotFoundError(
            f"Could not find {name}/, images+masks/, or {name}.zip under {dataset_dir} -- "
            f"run the /kaggle/input listing cell above and check what's actually there."
        )

    n_images = len(list((dst / "images").glob("*.jpg")))
    n_masks = len(list((dst / "masks").glob("*.png")))
    print(f"{name}: {n_images} images, {n_masks} masks staged at {dst}")


stage_tissue_data("aligned_raw", ALIGNED_RAW_DATASET_DIR)
stage_tissue_data("aligned_raw_forniceal", ALIGNED_RAW_FORNICEAL_DATASET_DIR)

aligned_raw: 201 images, 201 masks staged at Segmentation/data/processed/aligned_raw
aligned_raw_forniceal: 211 images, 211 masks staged at Segmentation/data/processed/aligned_raw_forniceal


In [7]:
# Combined sanity check, run BEFORE any real training:
#   1. Confirm the pip-installed heavy dependencies actually work -- a
#      dataloader-only check would NOT catch a missing/broken install here
#      (classification's own Kaggle notes: a plain dataloader check only
#      exercises dataset.py's imports, so a missing package silently
#      surfaces only much later, on the first real training script).
#   2. Confirm the 9-model registry itself imports cleanly.
#   3. Pull one real batch from BOTH tissue-type dataloaders.
import sys
from pathlib import Path

sys.path.insert(0, str(Path("Segmentation/scripts").resolve()))
sys.path.insert(0, str(Path("Segmentation").resolve()))

import segmentation_models_pytorch as smp
import timm
import transformers
import einops

print("segmentation_models_pytorch", smp.__version__)
print("timm", timm.__version__)
print("transformers", transformers.__version__)
print("einops", einops.__version__)

from models.segmentation.pretrained_registry import ARCHITECTURE_REGISTRY
print(f"\n{len(ARCHITECTURE_REGISTRY)} architectures registered:")
for name in ARCHITECTURE_REGISTRY:
    print(" ", name)

from dataset import get_dataloaders

loaders = get_dataloaders()
for key in ["aligned_seg_train", "aligned_seg_forniceal_train"]:
    images, masks = next(iter(loaders[key]))
    print(f"\n{key}: image batch {tuple(images.shape)}, mask batch {tuple(masks.shape)}, "
          f"{len(loaders[key].dataset)} patients")

segmentation_models_pytorch 0.5.0
timm 1.0.26
transformers 5.0.0
einops 0.8.2

9 architectures registered:
  cnn_base_efficientnet_b1_unet
  cnn_mid_resnet101_deeplabv3plus
  cnn_strong_convnext_large_unet
  hybrid_base_coatnet0_unet
  hybrid_mid_coatnet2_unet
  hybrid_strong_transunet
  transformer_base_segformer_b2
  transformer_mid_swin_base_upernet
  transformer_strong_swin_large_upernet

aligned_seg_train: image batch (16, 3, 256, 256), mask batch (16, 1, 256, 256), 143 patients

aligned_seg_forniceal_train: image batch (16, 3, 256, 256), mask batch (16, 1, 256, 256), 147 patients


## Training

In [8]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidates Segmentation/outputs/{checkpoints,logs,plots}/ into a single
    top-level /kaggle/working/outputs/ folder and re-zips it to
    /kaggle/working/segmentation_sweep_results.zip. Called after EVERY
    training cell below, not just at the end -- if the run gets cut short
    partway through the 18 combos, whatever completed so far is still
    cleanly consolidated and zipped, ready to download.

    IMPORTANT (added after a real Kaggle disk-quota crash, "Your notebook
    tried to use more disk space than is available", 20.93GB used, failed
    5h40m into a real run): each of the 18 training scripts can write up to
    3 full checkpoint files at fp32 (best-overall + one per loss function),
    and none of that is ever deleted between combos -- for the larger
    architectures (ConvNeXt-Large ~203M params, Swin-Large ~234M, etc.)
    those add up to hundreds of MB to ~1GB EACH. Originally this function
    copied Segmentation/outputs/ into /kaggle/working/outputs/ and left the
    source in place, so between the untouched source, the mirrored copy,
    and the zip made from that copy, roughly 3 copies of everything
    produced so far existed on disk simultaneously -- comfortably enough to
    blow a ~20GB quota partway through the Mid/Strong tiers. Now the
    source is deleted right after it's safely copied into
    /kaggle/working/outputs/, which becomes the single accumulating copy
    (plus the zip made from it) -- every training script recreates
    Segmentation/outputs/{checkpoints,logs,plots}/ fresh via its own
    mkdir(parents=True, exist_ok=True) on its next run, so nothing is lost
    by clearing the source here.
    """
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        src = Path("Segmentation/outputs") / sub
        if src.exists():
            shutil.copytree(src, results_dir / sub, dirs_exist_ok=True)
            shutil.rmtree(src)
    archive_path = shutil.make_archive("/kaggle/working/segmentation_sweep_results", "zip", root_dir=str(results_dir))
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

[sync_outputs] 0 files consolidated under /kaggle/working/outputs, zipped to /kaggle/working/segmentation_sweep_results.zip


### K-fold re-training — Base tier (4 combos)

In [9]:
# Base tier, combo 1/4 -- EfficientNet-B1 U-Net (CNN, ~8.8M), palpebral (3 folds, fixed best hyperparameters)
!python Segmentation/scripts/train_pretrained_kfold/train_kfold_cnn_base_efficientnet_b1_unet_palpebral.py
sync_outputs()

Using device: cuda
Model: cnn_base_efficientnet_b1_unet_palpebral (K-fold re-training, fixed hyperparameters)
tissue_type=palpebral image_size=256
learning_rate=0.0006358358856676254 weight_decay=0.000133112160807369 loss_fn=focal_tversky
n_folds=3 MAX_EPOCHS=250 EARLY_STOPPING_PATIENCE=7 BATCH_SIZE=32
Pool (train+val, aligned): 170 patients

=== cnn_base_efficientnet_b1_unet_palpebral | Fold 1/3: train=113 val=57 ===
config.json: 100%|██████████████████████████████| 106/106 [00:00<00:00, 640kB/s]
model.safetensors: 100%|███████████████████| 31.5M/31.5M [00:02<00:00, 15.4MB/s]
[cnn_base_efficientnet_b1_unet_palpebral | fold 1/3] Epoch   1/250 - train_loss=0.7015 val_loss=0.7577 val_dice=0.1242 val_iou=0.0667 lr=6.36e-04
[cnn_base_efficientnet_b1_unet_palpebral | fold 1/3] Epoch   2/250 - train_loss=0.6267 val_loss=0.7451 val_dice=0.1392 val_iou=0.0756 lr=6.36e-04
[cnn_base_efficientnet_b1_unet_palpebral | fold 1/3] Epoch   3/250 - train_loss=0.5681 val_loss=0.6807 val_dice=0.2512 val_i

In [10]:
# Base tier, combo 2/4 -- EfficientNet-B1 U-Net (CNN, ~8.8M), forniceal_palpebral (3 folds, fixed best hyperparameters)
!python Segmentation/scripts/train_pretrained_kfold/train_kfold_cnn_base_efficientnet_b1_unet_forniceal_palpebral.py
sync_outputs()

Using device: cuda
Model: cnn_base_efficientnet_b1_unet_forniceal_palpebral (K-fold re-training, fixed hyperparameters)
tissue_type=forniceal_palpebral image_size=256
learning_rate=0.0006358358856676254 weight_decay=0.000133112160807369 loss_fn=focal_tversky
n_folds=3 MAX_EPOCHS=250 EARLY_STOPPING_PATIENCE=7 BATCH_SIZE=32
Pool (train+val, aligned): 178 patients

=== cnn_base_efficientnet_b1_unet_forniceal_palpebral | Fold 1/3: train=118 val=60 ===
[cnn_base_efficientnet_b1_unet_forniceal_palpebral | fold 1/3] Epoch   1/250 - train_loss=0.5727 val_loss=0.6263 val_dice=0.1815 val_iou=0.1008 lr=6.36e-04
[cnn_base_efficientnet_b1_unet_forniceal_palpebral | fold 1/3] Epoch   2/250 - train_loss=0.4850 val_loss=0.5935 val_dice=0.2146 val_iou=0.1218 lr=6.36e-04
[cnn_base_efficientnet_b1_unet_forniceal_palpebral | fold 1/3] Epoch   3/250 - train_loss=0.4420 val_loss=0.5413 val_dice=0.3066 val_iou=0.1860 lr=6.36e-04
[cnn_base_efficientnet_b1_unet_forniceal_palpebral | fold 1/3] Epoch   4/250 - t

In [11]:
# Base tier, combo 3/4 -- CoAtNet-0 U-Net (Hybrid, ~30.8M), palpebral (3 folds, fixed best hyperparameters)
!python Segmentation/scripts/train_pretrained_kfold/train_kfold_hybrid_base_coatnet0_unet_palpebral.py
sync_outputs()

Using device: cuda
Model: hybrid_base_coatnet0_unet_palpebral (K-fold re-training, fixed hyperparameters)
tissue_type=palpebral image_size=224
learning_rate=0.0006358358856676254 weight_decay=0.000133112160807369 loss_fn=focal_tversky
n_folds=3 MAX_EPOCHS=250 EARLY_STOPPING_PATIENCE=7 BATCH_SIZE=32
Pool (train+val, aligned): 170 patients

=== hybrid_base_coatnet0_unet_palpebral | Fold 1/3: train=113 val=57 ===
model.safetensors: 100%|█████████████████████| 110M/110M [00:04<00:00, 27.3MB/s]
[hybrid_base_coatnet0_unet_palpebral | fold 1/3] Epoch   1/250 - train_loss=0.7861 val_loss=0.8010 val_dice=0.0998 val_iou=0.0529 lr=6.36e-04
[hybrid_base_coatnet0_unet_palpebral | fold 1/3] Epoch   2/250 - train_loss=0.6904 val_loss=0.7953 val_dice=0.1076 val_iou=0.0572 lr=6.36e-04
[hybrid_base_coatnet0_unet_palpebral | fold 1/3] Epoch   3/250 - train_loss=0.6544 val_loss=0.8075 val_dice=0.0869 val_iou=0.0457 lr=6.36e-04
[hybrid_base_coatnet0_unet_palpebral | fold 1/3] Epoch   4/250 - train_loss=0.6

In [12]:
# Base tier, combo 4/4 -- CoAtNet-0 U-Net (Hybrid, ~30.8M), forniceal_palpebral (3 folds, fixed best hyperparameters)
!python Segmentation/scripts/train_pretrained_kfold/train_kfold_hybrid_base_coatnet0_unet_forniceal_palpebral.py
sync_outputs()

Using device: cuda
Model: hybrid_base_coatnet0_unet_forniceal_palpebral (K-fold re-training, fixed hyperparameters)
tissue_type=forniceal_palpebral image_size=224
learning_rate=0.0001329291894316216 weight_decay=0.0007114476009343421 loss_fn=bce_dice
n_folds=3 MAX_EPOCHS=250 EARLY_STOPPING_PATIENCE=7 BATCH_SIZE=32
Pool (train+val, aligned): 178 patients

=== hybrid_base_coatnet0_unet_forniceal_palpebral | Fold 1/3: train=118 val=60 ===
[hybrid_base_coatnet0_unet_forniceal_palpebral | fold 1/3] Epoch   1/250 - train_loss=0.7829 val_loss=0.7751 val_dice=0.2334 val_iou=0.1337 lr=1.33e-04
[hybrid_base_coatnet0_unet_forniceal_palpebral | fold 1/3] Epoch   2/250 - train_loss=0.6914 val_loss=0.7169 val_dice=0.3279 val_iou=0.1997 lr=1.33e-04
[hybrid_base_coatnet0_unet_forniceal_palpebral | fold 1/3] Epoch   3/250 - train_loss=0.6346 val_loss=0.6661 val_dice=0.4116 val_iou=0.2652 lr=1.33e-04
[hybrid_base_coatnet0_unet_forniceal_palpebral | fold 1/3] Epoch   4/250 - train_loss=0.5958 val_loss=0.

## Done — what to download

This tier's fold checkpoints (fp16), logs (`*_kfold_summary.json`, `*_kfold_folds.csv`, `*_kfold_test_per_patient.csv`), and plots are consolidated at `/kaggle/working/outputs/` and zipped to `/kaggle/working/segmentation_sweep_results.zip`, visible in this notebook version's **Output** tab once you Save Version -> Save & Run All — download the zip directly from there.

A failed `!python ...` cell does **not** halt "Run All" — check each script's own printed output (or the saved `*_kfold_summary.json` files) after this finishes.

In [13]:
from pathlib import Path

print('Final contents of /kaggle/working/outputs:')
for f in sorted(Path('/kaggle/working/outputs').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/outputs")}  ({f.stat().st_size / 1e6:.2f} MB)')

zip_path = Path('/kaggle/working/segmentation_sweep_results.zip')
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")

Final contents of /kaggle/working/outputs:
  checkpoints/best_cnn_base_efficientnet_b1_unet_forniceal_palpebral_fold1.pth  (17.86 MB)
  checkpoints/best_cnn_base_efficientnet_b1_unet_forniceal_palpebral_fold2.pth  (17.86 MB)
  checkpoints/best_cnn_base_efficientnet_b1_unet_forniceal_palpebral_fold3.pth  (17.86 MB)
  checkpoints/best_cnn_base_efficientnet_b1_unet_palpebral_fold1.pth  (17.85 MB)
  checkpoints/best_cnn_base_efficientnet_b1_unet_palpebral_fold2.pth  (17.85 MB)
  checkpoints/best_cnn_base_efficientnet_b1_unet_palpebral_fold3.pth  (17.85 MB)
  checkpoints/best_hybrid_base_coatnet0_unet_forniceal_palpebral_fold1.pth  (61.73 MB)
  checkpoints/best_hybrid_base_coatnet0_unet_forniceal_palpebral_fold2.pth  (61.73 MB)
  checkpoints/best_hybrid_base_coatnet0_unet_forniceal_palpebral_fold3.pth  (61.73 MB)
  checkpoints/best_hybrid_base_coatnet0_unet_palpebral_fold1.pth  (61.73 MB)
  checkpoints/best_hybrid_base_coatnet0_unet_palpebral_fold2.pth  (61.73 MB)
  checkpoints/best_hybrid_